In [ ]:
import json
import random
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import amp
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as transforms
import torchvision.models as models

from scipy import stats
from sklearn.metrics import recall_score, f1_score

SEED = 42
ALPHA = 0.05
random.seed(SEED); np.random.seed(SEED)
rng = np.random.default_rng(SEED)

assert torch.cuda.is_available(), "CUDA 미탐지"
device = torch.device("cuda")

sns.set_theme(style="whitegrid")
plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

OUT = Path("outputs/garbage")
split     = pd.read_csv(OUT / "metrics" / "split.csv")
label_map = json.load(open(OUT / "metrics" / "label_map.json", encoding="utf-8"))
nstats    = json.load(open(OUT / "metrics" / "norm_stats.json", encoding="utf-8"))

classes   = [c for c, _ in sorted(label_map.items(), key=lambda kv: kv[1])]
N_CLASSES = len(classes)
train_df  = split[split.split == "train"].reset_index(drop=True)
val_df    = split[split.split == "val"].reset_index(drop=True)
test_df   = split[split.split == "test"].reset_index(drop=True)

print("scipy", stats.__name__, "| α =", ALPHA)
print(f"train {len(train_df):,} | val {len(val_df):,} | test {len(test_df):,}")